this fuction does the following:

1.   takes the path of an image as input covet to gray scale and reshape to 48*48

2.   then reshape again to (batch Size, hight, width ,channel )===>(1,48,48,1)



In [6]:
from keras.preprocessing import image
import numpy as np

In [1]:
def convert_face(image_path,target_size=(48,48),color_mode='grayscale'):
  img = load_img(image_path, target_size=target_size, color_mode=color_mode)
  img_array = img_to_array(img)
  return np.expand_dims(img_array, axis=0)


In [2]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import img_to_array

def capture_and_convert(camera_index=0, target_size=(48,48)):
    """
    Capture one image from the webcam, convert to grayscale, resize to target_size,
    and add batch dimension.

    Args:
        camera_index: usually 0 for built‑in webcam
        target_size: (height, width)

    Returns:
        numpy array of shape (1, height, width, 1), ready for model.predict()
    """
    # Open webcam
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError("Cannot access camera")

    # Capture a single frame
    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise RuntimeError("Failed to capture image")

    # Convert BGR (OpenCV default) to RGB, then to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Resize to target_size (width, height) for cv2.resize
    resized = cv2.resize(gray, (target_size[1], target_size[0]))

    # Add channel dimension -> (height, width, 1)
    img_array = np.expand_dims(resized, axis=-1)

    # Add batch dimension -> (1, height, width, 1)
    img_batch = np.expand_dims(img_array, axis=0)

    # Optional: normalize pixel values to [0,1]
    img_batch = img_batch / 255.0

    return img_batch

In [3]:
from google.colab import files
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import os

def upload_and_convert(target_size=(48,48), allowed_extensions=None):
    """
    Upload an image file from your computer, validate extension,
    then convert to 48x48 grayscale and add batch dimension.

    Args:
        target_size: (height, width) tuple, default (48,48)
        allowed_extensions: set of allowed extensions, e.g. {'.jpg', '.png'}
                            If None, defaults to common image extensions.

    Returns:
        numpy array of shape (1, height, width, 1) ready for model.predict()
    """
    if allowed_extensions is None:
        allowed_extensions = {'.jpg', '.jpeg', '.png'}

    print("Please upload an image file (valid extensions: {})".format(', '.join(allowed_extensions)))
    uploaded = files.upload()

    # Get the first uploaded filename
    filename = next(iter(uploaded.keys()))

    # Check file extension
    ext = os.path.splitext(filename)[1].lower()
    if ext not in allowed_extensions:
        raise ValueError(f"Invalid file type: {ext}. Allowed: {allowed_extensions}")

    # Load and preprocess the image
    img = load_img(filename, target_size=target_size, color_mode='grayscale')
    img_array = img_to_array(img)                # (48,48,1)
    img_batch = np.expand_dims(img_array, axis=0) # (1,48,48,1)
    img_batch = img_batch / 255.0                # normalize to [0,1]

    # Optional: remove the uploaded file to keep Colab clean
    os.remove(filename)

    return img_batch

In [ ]:
# Upload and get both the model‑ready batch and the original image
import matplotlib.pyplot as plt
img_batch= upload_and_convert()*255.0

# Plot the original image
plt.figure(figsize=(6,6))
plt.imshow(img_batch[0,:,:,0],cmap='gray')
plt.title("Uploaded image (original)")
plt.axis('off')
plt.show()

# Now use img_batch for prediction
# prediction = model.predict(img_batch)

Please upload an image file (valid extensions: .png, .jpg, .jpeg)
